# 04 — LoRA Fine-Tuned DeBERTa (Model of Choice)
**Roll No:** 23f3004491 | Model 4 of 5 | Milestone 4

The required fine-tuned model. The task is framed as multiple-choice: the model sees all five
(prompt, option) pairs together and a softmax picks the best. LoRA freezes the base weights and
trains only small adapter matrices, so it fits on a single GPU. **Requires a GPU.**

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{BASE}/train.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
OPTIONS = ["A", "B", "C", "D", "E"]

def average_precision_at_3(true_label, predicted_labels):
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

def mean_average_precision_at_3(true_labels, predicted_lists):
    return float(np.mean([average_precision_at_3(t, p)
                          for t, p in zip(true_labels, predicted_lists)]))

START_WRAPPERS = ["Pick the best possible answer:", "Select the most accurate option:",
                  "Determine the correct option:", "Identify the correct statement:",
                  "Choose the correct answer:"]

def normalize_core(prompt):
    p = str(prompt).strip()
    for s in START_WRAPPERS:
        if p.startswith(s):
            p = p[len(s):].strip()
    if "?" in p:
        p = p[:p.rfind("?") + 1]
    return re.sub(r"\s+", " ", p).lower().strip()

train["core"] = train["prompt"].apply(normalize_core)
test["core"]  = test["prompt"].apply(normalize_core)

np.random.seed(42)
cores = train["core"].unique().copy()
np.random.shuffle(cores)
val_cores = set(cores[:200])
valid_df = train[train["core"].isin(val_cores)].drop_duplicates("core").reset_index(drop=True)
print("train:", train.shape, "| validation questions:", len(valid_df))

In [ ]:
!pip install -q -U "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1"

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForMultipleChoice,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

assert torch.cuda.is_available(), "This notebook needs a GPU. Enable GPU T4 in Settings."

MODEL = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
label_map = {o: i for i, o in enumerate(OPTIONS)}

def encode(df):
    first  = sum([[p] * 5 for p in df["prompt"].astype(str)], [])
    second = sum([[str(r[o]) for o in OPTIONS] for _, r in df.iterrows()], [])
    tok = tokenizer(first, second, truncation=True, max_length=160, padding="max_length")
    grouped = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tok.items()}
    grouped["labels"] = [label_map[a] for a in df["answer"]]
    ds = Dataset.from_dict(grouped); ds.set_format("torch"); return ds

# train on the non-validation cores
train_df = train[~train["core"].isin(val_cores)].reset_index(drop=True)
train_ds, valid_ds = encode(train_df), encode(valid_df)

In [ ]:
base = AutoModelForMultipleChoice.from_pretrained(MODEL)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1,
                  target_modules=["query_proj", "key_proj", "value_proj"],
                  modules_to_save=["classifier", "pooler"])
model = get_peft_model(base, lora)
model.print_trainable_parameters()

def mc_metrics(eval_pred):
    logits, labels = eval_pred
    order = np.argsort(-logits, axis=1)
    ap3 = [1.0 / (np.where(o == t)[0][0] + 1) if t in o[:3] else 0.0
           for o, t in zip(order, labels)]
    return {"accuracy": float((order[:, 0] == labels).mean()), "map3": float(np.mean(ap3))}

args = TrainingArguments(output_dir="./mc_lora", num_train_epochs=2,
                         per_device_train_batch_size=8, per_device_eval_batch_size=16,
                         learning_rate=1e-4, warmup_ratio=0.1, eval_strategy="epoch",
                         save_strategy="no", report_to="none", fp16=True, seed=42)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=valid_ds, compute_metrics=mc_metrics)
trainer.train()

In [ ]:
logits = trainer.predict(valid_ds).predictions
preds = [[OPTIONS[j] for j in np.argsort(row)[::-1]] for row in logits]
score = mean_average_precision_at_3(valid_df["answer"].tolist(), preds)
print(f"LoRA DeBERTa validation MAP@3: {score:.4f}")

## Observation
The LoRA model reaches about 0.59 MAP@3 — the best of the trainable neural models, showing that
fine-tuning helps the model read prompt and options jointly. It is still below the retrieval
lookup, which exploits the data duplication directly.